# Simulação de Opinião Pública com LLMs

**Disciplina:** Inteligência Artificial, Mackenzie 2026/1  
**Professor:** Rogerio Oliveira  
**Grupo:** Matheus Henrique de Oliveira Santos e Marina Cantarelli Barroca

---

## Objetivo

Usar um LLM para simular respostas de uma pesquisa de opinião pública (Cesop 04839) e comparar com os dados reais.

Abordagem baseada em *silicon sampling* (Argyle et al.): o modelo recebe perfis sociodemográficos e responde como cada pessoa responderia.

## Notas sobre a implementação

**1. Otimização por lotes:** em vez de uma chamada de API por respondent que era muito lento, modifiquei o script pra enviar **lotes de 20 perfis por chamada**, reduzindo 200 chamadas para 10 por rodada.

**2. Mudança de provedor LLM:** o projeto foi originalmente implementado com Google Gemini (`gemini-2.5-flash-lite`), mas a quota do free tier estava restrita a apenas 20 requisições/dia, impedindo a execução completa. Migramos para **Groq** (modelo Llama 3.1 70B), que oferece quota gratuita mais generosa (30 req/min, sem limite diário rígido).

## 1. Setup

In [ ]:
!pip install -q groq pandas scikit-learn matplotlib seaborn

In [ ]:
from groq import Groq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix
import json, re, time

from google.colab import userdata


client = Groq(api_key=userdata.get('GROQ_KEY'))
MODEL_NAME = 'llama-3.3-70b-versatile'

print(f'OK - usando modelo {MODEL_NAME}')

## 2. Dados

Pesquisa **04839 - Percepção dos Brasileiros sobre Desigualdade** (Cesop/Unicamp, 2023), 1.849 respondentes após limpeza.

Questões simuladas:
- **Q1 (P6B):** Cotas nas universidades para negros e indígenas reduzem a desigualdade? (Concorda / Neutro / Discorda)
- **Q2 (P6E):** Mudanças climáticas afetam mais pessoas em situação de pobreza? (Concorda / Neutro / Discorda)

In [ ]:
import os
if not os.path.exists('dados_cesop_04839.csv'):
    from google.colab import files
    print('Faça upload do arquivo dados_cesop_04839.csv:')
    files.upload()

df = pd.read_csv('dados_cesop_04839.csv')
print(f'Dataset: {len(df)} respondentes')
df.head()

In [ ]:
# Distribuição real
ordem = ['Concorda', 'Neutro', 'Discorda']
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Distribuição Real - Cesop 04839')
df['q1_cotas'].value_counts(normalize=True).reindex(ordem).plot(
    kind='bar', ax=axes[0], color='steelblue', title='Q1: Cotas'
)
df['q2_clima'].value_counts(normalize=True).reindex(ordem).plot(
    kind='bar', ax=axes[1], color='steelblue', title='Q2: Clima'
)
for ax in axes:
    ax.set_ylabel('Proporção'); ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()

## 3. Simulação com LLM (em lotes)

Cada chamada à API processa 20 perfis numerados e retorna as respostas em JSON. Isso reduz drasticamente o número de chamadas e evita rate limiting.

In [ ]:
def montar_prompt_lote(grupo_df):
    """Monta prompt com vários perfis em uma única chamada."""
    perfis_txt = []
    for i, row in grupo_df.iterrows():
        idade = int(row['idade']) if not pd.isna(row['idade']) else 'NA'
        perfis_txt.append(
            f"{i}. Sexo:{row['sexo']}, {idade}a, {row['escolaridade']}, "
            f"Renda:{row['renda']}, {row['regiao']}"
        )
    perfis = '\n'.join(perfis_txt)
    
    return f"""Você vai simular respostas de pesquisa de opinião pública brasileira.
Para cada perfil abaixo, responda como essa pessoa responderia, considerando o contexto socioeconômico do Brasil.

PERFIS:
{perfis}

PERGUNTAS:
Q1: A maior presença de pessoas negras e indígenas nas universidades, por meio de cotas, é necessária para reduzir a desigualdade no Brasil. (Concorda / Neutro / Discorda)
Q2: As mudanças climáticas e eventos extremos afetam mais as pessoas em situação de pobreza. (Concorda / Neutro / Discorda)

RESPONDA APENAS no formato JSON válido abaixo, com uma entrada por perfil:
[
  {{"id": <numero_perfil>, "q1": "Concorda|Neutro|Discorda", "q2": "Concorda|Neutro|Discorda"}},
  ...
]
Não inclua explicações, markdown ou texto fora do JSON."""


def extrair_json(texto):
    """Extrai o JSON da resposta, tolerante a markdown."""
    texto = texto.strip()
    # remove markdown code fences se houver
    texto = re.sub(r'^```(?:json)?\s*', '', texto)
    texto = re.sub(r'\s*```$', '', texto)
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        # tenta pegar só o array com regex
        match = re.search(r'\[.*\]', texto, re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except:
                return []
        return []


def simular_rodada(df_full, seed=42, n=200, tamanho_lote=20):
    """Simula uma rodada completa em lotes usando Groq."""
    sub = df_full.sample(n, random_state=seed).reset_index(drop=True)
    sub['sim_q1'] = None
    sub['sim_q2'] = None
    
    validas = {'Concorda', 'Neutro', 'Discorda'}
    total_lotes = (len(sub) + tamanho_lote - 1) // tamanho_lote
    inicio = time.time()
    
    for i in range(0, len(sub), tamanho_lote):
        lote = sub.iloc[i:i+tamanho_lote]
        prompt = montar_prompt_lote(lote)
        
        try:
            resposta = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.7,
            )
            texto = resposta.choices[0].message.content
            dados = extrair_json(texto)
            
            if not dados:
                print(f'  [VAZIO] Lote {i//tamanho_lote + 1}: {texto[:200]}')
            
            for item in dados:
                idx = item.get('id')
                if idx is None or idx >= len(sub):
                    continue
                q1 = item.get('q1', '').strip()
                q2 = item.get('q2', '').strip()
                if q1 in validas:
                    sub.at[idx, 'sim_q1'] = q1
                if q2 in validas:
                    sub.at[idx, 'sim_q2'] = q2
        except Exception as e:
            print(f'  [ERRO] Lote {i//tamanho_lote + 1}: {type(e).__name__}: {e}')
        
        lote_num = i//tamanho_lote + 1
        print(f'  Lote {lote_num}/{total_lotes} ({time.time()-inicio:.0f}s)')
    
    return sub

print('Funções definidas.')

In [ ]:
# 3 rodadas com sementes diferentes
seeds = [42, 7, 13]
resultados = []

for s in seeds:
    print(f'\n=== Rodada seed={s} ===')
    res = simular_rodada(df, seed=s, n=200, tamanho_lote=20)
    validas = res['sim_q1'].notna().sum()
    print(f'Total válidas: {validas}/200')
    resultados.append(res)

print('\nSimulação concluída.')

## 4. Análise

In [ ]:
# Distribuição real vs simulada (primeira rodada)
rod1 = resultados[0].dropna(subset=['sim_q1', 'sim_q2'])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Distribuição Real vs Simulada', fontsize=14)
rod1['q1_cotas'].value_counts(normalize=True).reindex(ordem).plot(
    kind='bar', ax=axes[0,0], color='steelblue', title='Q1 - Real')
rod1['sim_q1'].value_counts(normalize=True).reindex(ordem).plot(
    kind='bar', ax=axes[0,1], color='coral', title='Q1 - Simulado')
rod1['q2_clima'].value_counts(normalize=True).reindex(ordem).plot(
    kind='bar', ax=axes[1,0], color='steelblue', title='Q2 - Real')
rod1['sim_q2'].value_counts(normalize=True).reindex(ordem).plot(
    kind='bar', ax=axes[1,1], color='coral', title='Q2 - Simulado')
for ax in axes.flat:
    ax.set_ylabel('Proporção'); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30); ax.set_ylim(0, 1)
plt.tight_layout(); plt.show()

In [ ]:
# Acurácia por rodada
acc_q1, acc_q2 = [], []
print('Acurácia por rodada:\n')
for i, rodada in enumerate(resultados):
    r = rodada.dropna(subset=['sim_q1', 'sim_q2'])
    if len(r) == 0:
        print(f'  Rodada {i+1} (seed={seeds[i]}): sem respostas válidas')
        continue
    a1 = accuracy_score(r['q1_cotas'], r['sim_q1'])
    a2 = accuracy_score(r['q2_clima'], r['sim_q2'])
    acc_q1.append(a1); acc_q2.append(a2)
    print(f'  Rodada {i+1} (seed={seeds[i]}): Q1={a1:.2%} | Q2={a2:.2%}')

if acc_q1:
    print(f'\nMédia Q1: {np.mean(acc_q1):.2%} (±{np.std(acc_q1):.2%})')
    print(f'Média Q2: {np.mean(acc_q2):.2%} (±{np.std(acc_q2):.2%})')

In [ ]:
# Matriz de confusão Q1
rod_cm = resultados[0].dropna(subset=['sim_q1'])
cm = confusion_matrix(rod_cm['q1_cotas'], rod_cm['sim_q1'], labels=ordem)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=ordem, yticklabels=ordem, cmap='Blues')
plt.title('Matriz de Confusão - Q1')
plt.ylabel('Real'); plt.xlabel('Simulado')
plt.tight_layout(); plt.show()

In [ ]:
# Acurácia Q1 por escolaridade
rod_esc = resultados[0].dropna(subset=['sim_q1'])
grupos_dict = {}
for nome, grupo in rod_esc.groupby('escolaridade'):
    grupos_dict[nome] = accuracy_score(grupo['q1_cotas'], grupo['sim_q1'])

grupos = pd.DataFrame(list(grupos_dict.items()), columns=['Escolaridade', 'Acurácia'])
grupos = grupos.sort_values('Acurácia', ascending=False)

plt.figure(figsize=(8, 4))
plt.bar(grupos['Escolaridade'], grupos['Acurácia'], color='steelblue')
plt.title('Acurácia Q1 por Escolaridade'); plt.ylabel('Acurácia')
plt.ylim(0, 1); plt.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.show()
print(grupos.to_string(index=False))

In [ ]:
# Acurácia Q1 por região
rod_reg = resultados[0].dropna(subset=['sim_q1'])
regioes_dict = {}
for nome, grupo in rod_reg.groupby('regiao'):
    regioes_dict[nome] = accuracy_score(grupo['q1_cotas'], grupo['sim_q1'])

por_regiao = pd.DataFrame(list(regioes_dict.items()), columns=['Região', 'Acurácia'])
por_regiao = por_regiao.sort_values('Acurácia', ascending=False)

plt.figure(figsize=(8, 4))
plt.bar(por_regiao['Região'], por_regiao['Acurácia'], color='teal')
plt.title('Acurácia Q1 por Região'); plt.ylabel('Acurácia')
plt.ylim(0, 1); plt.tight_layout(); plt.show()
print(por_regiao.to_string(index=False))

## 5. Discussão

**[Preencher após rodar com os resultados reais]**

Pontos a discutir:
- A distribuição simulada se aproximou da real? Em qual questão o modelo foi mais preciso?
- Algum grupo (escolaridade, região) teve acurácia diferente? Por quê?
- Qual foi a variância entre as 3 repetições?
- Limitações: vieses do treinamento do LLM, comportamento da classe majoritária.

## 6. Conclusão

**[Preencher após análise]**

## Notas sobre a Implementação

Este notebook usa **Groq** com o modelo Llama 3.1 70B como provedor de LLM. A escolha foi feita após uma tentativa inicial com Google Gemini (`gemini-2.5-flash-lite`), que apresentou limitação severa de quota no free tier (20 requisições/dia em vez das 1.000 documentadas), inviabilizando a execução completa do experimento.

**Comparação dos provedores testados:**

| Provedor | Modelo | Quota Free Tier | Status no Projeto |
|---|---|---|---|
| Google Gemini | gemini-2.5-flash-lite | 20 req/dia (observado) | Inviável |
| Groq | llama-3.3-70b-versatile | 30 req/min, ~14.400/dia | Utilizado |

## Referências

ARGYLE, L. P. et al. *Simulating Public Opinion: Comparing Distributional and Individual-Level Predictions from LLMs and Random Forests*.

CESOP/UNICAMP. *04839 - Percepção dos Brasileiros sobre Desigualdade*, 2023. https://www.cesop.unicamp.br

GROQ. *API Documentation*. https://console.groq.com/docs